# Sentiment Analysis using Sentence Transformers

Pipeline:

1. Load `data_split.csv`
2. Use existing train/val/test split
3. Generate embeddings with Hugging Face Sentence Transformers
4. Tune Logistic Regression on validation set
5. Tune Linear SVM on validation set
6. Evaluate on test set using Macro F1
7. Save best model


In [ ]:
!pip install -q sentence-transformers

In [4]:
import re
import warnings
import numpy as np
import pandas as pd
import pickle
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report

SEED = 42

data = pd.read_csv("data_split.csv", index_col=0)
data["clean_eda"] = data["clean_eda"].fillna("").astype(str)
data["sentiment"] = data["sentiment"].astype(int)

train_df = data[data["split"] == "train"].reset_index(drop=True)
val_df   = data[data["split"] == "val"].reset_index(drop=True)
test_df  = data[data["split"] == "test"].reset_index(drop=True)

print(f"train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")
print("Class balance (train):")
print(train_df["sentiment"].value_counts(normalize=True).sort_index().round(3))
print("Columns:", data.columns.tolist())

train: 178,557  val: 38,263  test: 38,263
Class balance (train):
sentiment
-1    0.352
 0    0.249
 1    0.399
Name: proportion, dtype: float64
Columns: ['Text', 'clean_eda', 'sentiment', 'Type', 'split']


In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding train set using raw Text...")
X_train = model.encode(
    train_df["Text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Encoding val set...")
X_val = model.encode(
    val_df["Text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Encoding test set...")
X_test = model.encode(
    test_df["Text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

y_train = train_df["sentiment"].to_numpy()
y_val   = val_df["sentiment"].to_numpy()
y_test  = test_df["sentiment"].to_numpy()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

# Save with different name so you keep the old ones too
np.save("X_train_minilm_rawtext.npy", X_train)
np.save("X_val_minilm_rawtext.npy",   X_val)
np.save("X_test_minilm_rawtext.npy",  X_test)
print("Saved!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding train set using raw Text...


Batches:   0%|          | 0/2790 [00:00<?, ?it/s]

Encoding val set...


Batches:   0%|          | 0/598 [00:00<?, ?it/s]

Encoding test set...


Batches:   0%|          | 0/598 [00:00<?, ?it/s]

X_train shape: (178557, 384)
X_val shape:   (38263, 384)
X_test shape:  (38263, 384)
Saved!


In [6]:
from sentence_transformers import SentenceTransformer

print("Encoding train set... (biggest, takes ~10 min)")
X_train = model.encode(
    train_df["clean_eda"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Encoding val set...")
X_val = model.encode(
    val_df["clean_eda"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Encoding test set...")
X_test = model.encode(
    test_df["clean_eda"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

y_train = train_df["sentiment"].to_numpy()
y_val   = val_df["sentiment"].to_numpy()
y_test  = test_df["sentiment"].to_numpy()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

# Save so you never have to encode again
np.save("X_train_minilm.npy", X_train)
np.save("X_val_minilm.npy",   X_val)
np.save("X_test_minilm.npy",  X_test)
print("Embeddings saved!")

Encoding train set... (biggest, takes ~10 min)


Batches:   0%|          | 0/2790 [00:00<?, ?it/s]

Encoding val set...


Batches:   0%|          | 0/598 [00:00<?, ?it/s]

Encoding test set...


Batches:   0%|          | 0/598 [00:00<?, ?it/s]

X_train shape: (178557, 384)
X_val shape:   (38263, 384)
X_test shape:  (38263, 384)
Embeddings saved!


In [14]:
# Load new embeddings
X_train = np.load("X_train_minilm_rawtext.npy")
X_val   = np.load("X_val_minilm_rawtext.npy")
X_test  = np.load("X_test_minilm_rawtext.npy")

y_train = train_df["sentiment"].to_numpy()
y_val   = val_df["sentiment"].to_numpy()
y_test  = test_df["sentiment"].to_numpy()

print(f"Loaded: {X_train.shape}, {X_val.shape}, {X_test.shape}")

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)
print("Scaled.")

Loaded: (178557, 384), (38263, 384), (38263, 384)
Scaled.


In [15]:
print("Tuning C for LogReg...\n")

best_c, best_f1 = 1.0, 0.0
for C in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    lr_c = LogisticRegression(
        C=C, max_iter=1000,
        class_weight={-1: 1.0, 0: 1.5, 1: 1.0},
        random_state=SEED
    )
    lr_c.fit(X_train_s, y_train)
    preds = lr_c.predict(X_val_s)
    f1 = f1_score(y_val, preds, average="macro")
    print(f"  C={C}  ->  F1={f1:.4f}")
    if f1 > best_f1:
        best_f1, best_c = f1, C

print(f"\nBest C: {best_c}  (F1={best_f1:.4f})")

Tuning C for LogReg...

  C=0.1  ->  F1=0.6372
  C=0.5  ->  F1=0.6369
  C=1.0  ->  F1=0.6372
  C=2.0  ->  F1=0.6370
  C=5.0  ->  F1=0.6372
  C=10.0  ->  F1=0.6371

Best C: 0.1  (F1=0.6372)


In [16]:
print("Tuning C for LinearSVC...\n")

best_c_svc, best_f1_svc = 1.0, 0.0
for C in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    svc_c = LinearSVC(
        C=C, max_iter=2000,
        class_weight={-1: 1.0, 0: 1.5, 1: 1.0},
        random_state=SEED
    )
    svc_c.fit(X_train_s, y_train)
    preds = svc_c.predict(X_val_s)
    f1 = f1_score(y_val, preds, average="macro")
    print(f"  C={C}  ->  F1={f1:.4f}")
    if f1 > best_f1_svc:
        best_f1_svc, best_c_svc = f1, C

print(f"\nBest C for SVC: {best_c_svc}  (F1={best_f1_svc:.4f})")
print(f"Best C for LR:  {best_c}  (F1={best_f1:.4f})")
print(f"\nBest overall: {'LogReg' if best_f1 > best_f1_svc else 'LinearSVC'}")

Tuning C for LinearSVC...

  C=0.1  ->  F1=0.6309
  C=0.5  ->  F1=0.6309
  C=1.0  ->  F1=0.6309
  C=2.0  ->  F1=0.6309
  C=5.0  ->  F1=0.6309
  C=10.0  ->  F1=0.6309

Best C for SVC: 0.5  (F1=0.6309)
Best C for LR:  0.1  (F1=0.6372)

Best overall: LogReg


In [17]:
# Pick best classifier
if best_f1 >= best_f1_svc:
    best_clf_name = "LogReg"
    final_clf = LogisticRegression(
        C=best_c, max_iter=1000,
        class_weight={-1: 1.0, 0: 1.5, 1: 1.0},
        random_state=SEED
    )
else:
    best_clf_name = "LinearSVC"
    final_clf = LinearSVC(
        C=best_c_svc, max_iter=2000,
        class_weight={-1: 1.0, 0: 1.5, 1: 1.0},
        random_state=SEED
    )

# Retrain on train+val combined
X_trval_s = np.vstack([X_train_s, X_val_s])
y_trval   = np.concatenate([y_train, y_val])
final_clf.fit(X_trval_s, y_trval)

y_pred_test = final_clf.predict(X_test_s)

print("=" * 60)
print(f"FINAL TEST — Route B (MiniLM raw text + {best_clf_name})")
print("=" * 60)
macro_f1 = f1_score(y_test, y_pred_test, average="macro")
print(f"Test macro-F1: {macro_f1:.4f}\n")
print(classification_report(
    y_test, y_pred_test,
    target_names=["Negative", "Neutral", "Positive"]
))

FINAL TEST — Route B (MiniLM raw text + LogReg)
Test macro-F1: 0.6398

              precision    recall  f1-score   support

    Negative       0.71      0.69      0.70     13468
     Neutral       0.46      0.51      0.48      9528
    Positive       0.76      0.73      0.74     15267

    accuracy                           0.66     38263
   macro avg       0.64      0.64      0.64     38263
weighted avg       0.67      0.66      0.66     38263

